# 3DModelExoplanet.ipynb

**Data sources (ONLY these three files are used):**
- `PS.csv`
- `ExoMinerals_Prototype.ipynb`
- `ExoMinerals_Prototype_(DepotReadinessScoreCalculated).ipynb`

This notebook performs the following, strictly using the files above:

1. **3D Model of All Exoplanets** Created a 3D interactive model based on distance given in PS.csv .
2. **3D Model of Only Top 25 Exoplanets** Defined and CSV created in `ExoMinerals_Prototype_(DepotReadinessScoreCalculated).ipynb`.
   - **Resource Richness** calculated from the ExoMinerals_Prototype.ipynb.
   - **Mass** from `PS.csv`.
   - **Celestial Body Type** calculated from various metrics from `PS.csv`.
4. For **each measured resource** in ExoMinerals_Prototype.ipynb, plot a different 3D Model with the 25 exoplanets.
5. **Categorize each of the 25** by the resource in which they are the most dominant.




In [210]:

# Imports
import os, json, math, re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from IPython.display import display, HTML

# Files
PS_PATH  = "PS.csv"
NB_MIN_A = "ExoMinerals_Prototype.ipynb"
NB_MIN_B = "ExoMinerals_Prototype_(DepotReadinessScoreCalculated).ipynb"



## 1) Load `PS.csv` and create a 3D model of all of the Exoplanets after finding their Cartesian coordinates

In [211]:
# Load PS.csv and drop any exoplanets that do not have RA or Dec fully
ps = pd.read_csv(PS_PATH, comment="#")
ps = ps.dropna(subset=["ra","dec"]).reset_index(drop=True)

### 3D model of all exoplanets 

In [212]:

# Convert from degrees to radians
ra  = np.deg2rad(ps["ra"].values)
dec = np.deg2rad(ps["dec"].values)

# Convert spherical to cartesian coordinates and convert parsecs to light years
x = d * np.cos(dec) * np.cos(ra)
y = d * np.cos(dec) * np.sin(ra)
z = d * np.sin(dec)
pc_to_ly = 3.26156

# Fill in unknowns and create the 3D Model
if np.isfinite(dist_pc).any():
    inv = 1.0 / (dist_pc.fillna(dist_pc[dist_pc.notna()].median()) + 1e-6)
    inv_norm = (inv - inv.min()) / (inv.max() - inv.min() + 1e-9)
    sizes = 2 + 8 * inv_norm
else:
    sizes = np.full(len(ps), 5.0)

hover = [
    f"RA: {ra_deg:.2f}° | Dec: {dc_deg:.2f}° | Dist: "
    + ("unknown" if pd.isna(pc) else f"{pc:.1f} pc / {pc*pc_to_ly:.1f} ly")
    for ra_deg, dc_deg, pc in zip(ps['ra'], ps['dec'], dist_pc)
]

fig_all = go.Figure()
fig_all.add_trace(go.Scatter3d(
    x=x, y=y, z=z, mode="markers",
    marker=dict(size=sizes, color=dist_pc, colorscale="Viridis", showscale=True, colorbar=dict(title="Distance (pc)")),
    text=hover, hoverinfo="text", name="Exoplanets"
))
# Earth is at the origin
fig_all.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers+text",
    marker=dict(size=6, color="black"), text=["Earth"], textposition="bottom center", name="Earth"
))
fig_all.update_layout(
    title="All Exoplanets — 3D Interactive Model with XYZ Coordinates",
    scene=dict(xaxis_title="X ", yaxis_title="Y ", zaxis_title="Z "),
    width=950, height=800
)
fig_all.show()


## 2) 3D Interactive Model of the 25 "best" exoplanets based on various factors (resource richness, mass, type of body)

In [213]:
# Convert to light years and get exoplanet and host names from csv of 25 "best" exoplanets
pc_to_ly = 3.26156
names = top25.get("pl_name", pd.Series(["(unknown)"]*len(top25)))
hosts = top25.get("hostname", pd.Series(["(unknown)"]*len(top25)))


#Creates the pop-up on the interactive model
hover25 = [
    f"<b>{n}</b><br>Host: {h}<br>RA: {ra:.2f}°  Dec: {dc:.2f}°"
    f"<br>Distance: {('unknown' if pd.isna(pc) else f'{pc:.1f} pc / {pc*pc_to_ly:.1f} ly')}"
    f"<br>EMI: {emi:.3f}"
    for n,h,ra,dc,pc,emi in zip(names, hosts, top25['ra'], top25['dec'], top25['sy_dist'], top25['EMI'])
]

# Format the 3D Interactive Model
fig_emi = go.Figure()
fig_emi.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers+text",
    marker=dict(size=6, color="black"), text=["Earth"], textposition="bottom center", name="Earth"
))
fig_emi.add_trace(go.Scatter3d(
    x=x25, y=y25, z=z25, mode="markers",
    marker=dict(size=10, color=top25["EMI"], colorscale="Turbo", showscale=True, colorbar=dict(title="EMI")),
    text=hover25, hoverinfo="text", name="Top-25 by EMI"
))
fig_emi.update_layout(
    title="25 Exoplanets — 3D Interactive Model based on Resource Richness",
    scene=dict(xaxis_title="X (pc)", yaxis_title="Y (pc)", zaxis_title="Z (pc)"),
    width=950, height=800
)
fig_emi.show()


In [214]:
# Find mass from csv of 25 "best" exoplanets
mass = pd.to_numeric(top25.get("pl_bmasse"), errors="coerce")


# Format the 3D Interactive Model based on Mass
fig_mass = go.Figure()
fig_mass.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers+text",
    marker=dict(size=6, color="black"), text=["Earth"], textposition="bottom center", name="Earth"
))
fig_mass.add_trace(go.Scatter3d(
    x=x25, y=y25, z=z25, mode="markers",
    marker=dict(
        size=10,
        color=mass,
        colorscale="Plasma",
        showscale=True,
        colorbar=dict(title="Mass (Earth masses)")
    ),
    text=hover25, hoverinfo="text", name="Top-25 (mass color)"
))
fig_mass.update_layout(
    title="25 Exoplanets — 3D Interactive Model based on Mass",
    scene=dict(xaxis_title="X (pc)", yaxis_title="Y (pc)", zaxis_title="Z (pc)"),
    width=950, height=800
)
fig_mass.show()


In [215]:
# Classify exoplanets into different types
def classify_row(r):
    R = r.get("pl_rade")
    M = r.get("pl_bmasse")
    D = r.get("pl_dens")

    if pd.notna(D):
        if D < 1.5:
            return "Gas Giant"
        if D >= 4.0:
            return "Rocky"

    if pd.notna(R) and pd.notna(M):
        if (R <= 1.6 and M <= 10):
            return "Rocky"
        if (R > 4.0) or (M >= 50):
            return "Gas Giant"
        if 1.6 < R <= 4.0:
            return "Mini/Sub-Neptune"
        if 10 < M < 50:
            return "Mini/Sub-Neptune"
        return "Unknown"
    if pd.notna(R):
        if R <= 1.6: return "Rocky"
        if R > 4.0:  return "Gas Giant"
        if 1.6 < R <= 4.0: return "Mini/Sub-Neptune"
    if pd.notna(M):
        if M >= 50: return "Gas Giant"
        if M <= 10: return "Rocky"
        return "Mini/Sub-Neptune"
    return "Unknown"

top25["type"] = top25.apply(classify_row, axis=1)
type_colors = {"Rocky":"#FF7F0E","Mini/Sub-Neptune":"#1F77B4","Gas Giant":"#2CA02C","Unknown":"#7F7F7F"}

# Create interactive 3D Model based on calculated Body Type
fig_type = go.Figure()
fig_type.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers+text",
    marker=dict(size=6, color="black"), text=["Earth"], textposition="bottom center", name="Earth"
))

for t, color in type_colors.items():
    m = (top25["type"] == t).values
    if m.any():
        fig_type.add_trace(go.Scatter3d(
            x=x25[m], y=y25[m], z=z25[m], mode="markers",
            marker=dict(size=10, color=color),
            text=[f"{n} — {t}" for n in top25.get("pl_name", pd.Series(["(unknown)"]*len(top25)))[m]],
            hoverinfo="text",
            name=t
        ))

fig_type.update_layout(
    title="25 Exoplanets — 3D Interactive Model based on type of Celestial Body",
    scene=dict(xaxis_title="X (pc)", yaxis_title="Y (pc)", zaxis_title="Z (pc)"),
    width=950, height=800
)
fig_type.show()


## 3) 3D Interactive Model of the 25 "best" exoplanets based on specific resource potential on separate models (Iron, Water, Silicates, Sulfates/Carbonates)

In [216]:
# Get Coordinates
def _ensure_xyz_from_top25(df):
    if all(v in globals() for v in ["x25", "y25", "z25"]):
        return globals()["x25"], globals()["y25"], globals()["z25"]
    assert {"ra","dec"}.issubset(df.columns), "top25 must contain 'ra' and 'dec'"
    ra  = np.deg2rad(df["ra"].values)
    dec = np.deg2rad(df["dec"].values)
    dpc = pd.to_numeric(df.get("sy_dist"), errors="coerce").fillna(1.0).values
    x = dpc * np.cos(dec) * np.cos(ra)
    y = dpc * np.cos(dec) * np.sin(ra)
    z = dpc * np.sin(dec)
    return x, y, z

x25, y25, z25 = _ensure_xyz_from_top25(top25)

# Display name per point
if "pl_name" in top25.columns:
    _names = top25["pl_name"].astype(str)
elif "hostname" in top25.columns:
    _names = top25["hostname"].astype(str)
else:
    _names = pd.Series([f"obj-{i}" for i in range(len(top25))])

# Pick best column for each resource
RESOURCE_BASES = [
    "pred_iron_metal",
    "pred_silicates",
    "pred_water_ice",
    "pred_sulfates_carbonates",
]

def pick_column(df: pd.DataFrame, base: str):
    for suffix in ["_max", "_mean", ""]:
        cand = base + suffix
        if cand in df.columns:
            return cand
    return None

resource_map = {}  
for base in RESOURCE_BASES:
    col = pick_column(top25, base)
    if col is not None:
        resource_map[base] = col

# Scale resource values in order to compare properly

def robust_minmax(s):
    s = pd.to_numeric(s, errors="coerce").astype(float)
    if s.notna().sum() == 0:
        return np.zeros_like(s)
    q5, q95 = np.nanpercentile(s, 5), np.nanpercentile(s, 95)
    if np.isfinite(q95 - q5) and (q95 > q5):
        x = (s - q5) / (q95 - q5)
    elif np.isfinite(s.min()) and np.isfinite(s.max()) and s.max() > s.min():
        x = (s - s.min()) / (s.max() - s.min())
    else:
        x = np.zeros_like(s)
    return np.clip(x, 0, 1)

# Make the 3D Column for each resource
for base, col in resource_map.items():
    vals = robust_minmax(top25[col])
    fig_res = go.Figure()
    fig_res.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode="markers+text",
        marker=dict(size=6, color="black"),
        text=["Earth"], textposition="bottom center",
        name="Earth"
    ))
# Make color show intensity of each resource
    fig_res.add_trace(go.Scatter3d(
        x=x25, y=y25, z=z25,
        mode="markers",
        marker=dict(size=10, color=vals, colorscale="Viridis", showscale=True,
                    colorbar=dict(title=col)),
        text=[f"{n} — {col}: {v:.3f}" for n, v in zip(_names, vals)],
        hoverinfo="text",
        name=col
    ))
    fig_res.update_layout(
        title=f"25 Exoplanets — 3D Interactive Model based on {col}",
        scene=dict(xaxis_title="X (pc)", yaxis_title="Y (pc)", zaxis_title="Z (pc)"),
        width=950, height=800
    )
    fig_res.show()

## 4) 3D Interactive Model of the 25 "best" exoplanets based on what resource they have the most potential in

In [217]:
sub = top25.copy()

# Get Coordinates
def _ensure_xyz_from_top25(df):
    if all(v in globals() for v in ["x25", "y25", "z25"]):
        return globals()["x25"], globals()["y25"], globals()["z25"]
    assert {"ra","dec"}.issubset(df.columns), "top25 must contain 'ra' and 'dec'"
    ra  = np.deg2rad(df["ra"].values)
    dec = np.deg2rad(df["dec"].values)
    dpc = pd.to_numeric(df.get("sy_dist"), errors="coerce").fillna(1.0).values
    x = dpc * np.cos(dec) * np.cos(ra)
    y = dpc * np.cos(dec) * np.sin(ra)
    z = dpc * np.sin(dec)
    return x, y, z

x25, y25, z25 = _ensure_xyz_from_top25(sub)

# Display name per point
if "pl_name" in sub.columns:
    _names = sub["pl_name"].astype(str)
elif "hostname" in sub.columns:
    _names = sub["hostname"].astype(str)
else:
    _names = pd.Series([f"obj-{i}" for i in range(len(sub))])

# Pick best column per resource
RESOURCE_LABEL_TO_BASE = {
    "Iron metal": "pred_iron_metal",
    "Silicates": "pred_silicates",
    "Water ice": "pred_water_ice",
    "Sulfates/Carbonates": "pred_sulfates_carbonates",
}
SUFFIX_PREF = ["_max", "_mean", ""]  

def pick_col(df, base):
    for suf in SUFFIX_PREF:
        cand = base + suf
        if cand in df.columns:
            return cand
    fallback = {
        "pred_iron_metal": "iron_metal",
        "pred_silicates": "silicates",
        "pred_water_ice": "water_ice",
        "pred_sulfates_carbonates": "sulfates_carbonates",
    }[base]
    return fallback if fallback in df.columns else None

avail_map = {label: pick_col(sub, base) for label, base in RESOURCE_LABEL_TO_BASE.items()}
avail_map = {label: col for label, col in avail_map.items() if col is not None}

# Scale resource values in order to compare properly
def robust_minmax(s):
    s = pd.to_numeric(s, errors="coerce").astype(float)
    if s.notna().sum() == 0:
        return np.zeros_like(s)
    q5, q95 = np.nanpercentile(s, 5), np.nanpercentile(s, 95)
    if np.isfinite(q95 - q5) and (q95 > q5):
        x = (s - q5) / (q95 - q5)
    elif np.isfinite(s.min()) and np.isfinite(s.max()) and s.max() > s.min():
        x = (s - s.min()) / (s.max() - s.min())
    else:
        x = np.zeros_like(s)
    return np.clip(x, 0, 1)

scaled = pd.DataFrame({label: robust_minmax(sub[col]) for label, col in avail_map.items()}, index=sub.index)
dominant = scaled.idxmax(axis=1)  

# Pick which colors map to which resource
cat_colors = {
    "Iron metal": "#d62728",           
    "Silicates": "#1f77b4",            
    "Water ice": "#17becf",            
    "Sulfates/Carbonates": "#2ca02c",  
    "Unknown": "#7f7f7f"               
}

# Build 3D Model and make sure that there is a dominant resource for each one.
fig_dom = go.Figure()

fig_dom.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers+text",
    marker=dict(size=6, color="black"),
    text=["Earth"], textposition="bottom center",
    name="Earth"
))

for cat in sorted(dominant.unique()):
    mask = (dominant == cat).values
    color = cat_colors.get(cat, "#7f7f7f")
    vals_text = []
    for i in np.where(mask)[0]:
        parts = [f"<b>{_names.iloc[i]}</b>", f"Dominant: {cat}"]
        for label, col in avail_map.items():
            v = sub[col].iloc[i]
            if pd.notna(v):
                parts.append(f"{label}: {float(v):.3f}")
        vals_text.append("<br>".join(parts))

    fig_dom.add_trace(go.Scatter3d(
        x=x25[mask], y=y25[mask], z=z25[mask],
        mode="markers",
        marker=dict(size=10, color=color),
        text=vals_text, hoverinfo="text",
        name=cat
    ))

fig_dom.update_layout(
    title="25 Exoplanets — 3D Interactive Model based on Dominant Resource",
    scene=dict(xaxis_title="X (pc)", yaxis_title="Y (pc)", zaxis_title="Z (pc)"),
    width=950, height=800
)

fig_dom.show()